# 64 — Precision & Recall for NLP
**Goal:** Measure entity extraction accuracy with precision, recall, and F1.

A skill extractor that "finds skills" is useless until you measure how often it is right. This chapter introduces the three canonical extraction metrics — **precision**, **recall**, and **F1** — and applies them to the resume-skill use case, first as aggregate scores, then broken down per skill category.

**Why it matters for resumes / ATS:** an ATS acts on extracted skills (filtering, matching, ranking), so every extraction error has a real cost. A **false positive** invents a skill the candidate never claimed — a hallucinated "Docker" can surface a candidate who cannot back it up. A **false negative** drops a real skill ("Java") and silently hides an otherwise strong match. Choosing whether to prioritize precision or recall is a product decision: conservative screening prefers high precision, broad matching prefers high recall, and F1 gives you one number to optimize when both matter.

## 1. Why Metrics Matter

A naive "how many skills did we find" count conflates two very different failure modes: finding *wrong* things and missing *right* ones. **Precision** answers "of everything we extracted, how much was correct?" while **recall** answers "of everything that was actually there, how much did we find?" **F1** is their harmonic mean — it stays low if *either* is low, so it punishes systems that cheat by extracting everything (max recall, terrible precision) or nothing (perfect precision, zero recall).

**What the code does:** the first cell prints the worked example used throughout the chapter — a resume with `[Python, Java, TensorFlow]`, a system that returns `[Python, TensorFlow, Docker]`, and the arithmetic: 2 of 3 returned skills are correct (precision 0.67) and 2 of 3 real skills are found (recall 0.67). The `Docker` mention is the false positive; the missed `Java` is the false negative.

**Try it:** change the example — a system that returns only `[Python]` has precision 1.0 but recall 0.33: perfect precision is easy, and useless alone.

In [ ]:
print('''For resume NLP, we care about:
- Precision: Of the skills we found, how many were correct?
- Recall: Of the actual skills, how many did we find?
- F1: Harmonic mean of both

Example: Resume has [Python, Java, TensorFlow]
System finds: [Python, TensorFlow, Docker]
Precision = 2/3 = 0.67 (Docker was wrong)
Recall = 2/3 = 0.67 (missed Java)
F1 = 0.67''')

## 2. Computing Metrics

`compute_metrics()` is the reusable core of the chapter. It converts both lists to sets, then counts the three confusion cells with pure set algebra: `tp` = intersection, `fp` = predicted-minus-gold, `fn` = gold-minus-predicted. The guard clauses (`if (tp + fp) > 0`, etc.) matter — precision and recall are undefined when there is nothing to divide, and a division by zero would crash a batch run on exactly the edge cases you most want to measure.

**What the code does:** the test loop runs five deliberately chosen cases — empty/empty, perfect match, a partial miss, a resume with no prediction, and an empty gold set with a spurious prediction. Running it reports: perfect → P=1.00 R=1.00 F1=1.00; partial (`gold [Python, Java, SQL]` vs `[Python, TensorFlow]`) → P=0.50, R=0.33, F1=0.40 — note F1 sits *below* both, the harmonic-mean penalty; the empty and one-sided cases all floor at 0.00, a correct non-crashing answer for degenerate input.

**Try it:** the "hallucination" case (empty gold, one prediction) is exactly the failure mode of an LLM that invents skills for a resume that lists none.

In [ ]:
def compute_metrics(gold, predicted):
    """Compute precision, recall, F1 for entity extraction."""
    gold_set = set(gold)
    pred_set = set(predicted)
    
    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}

# Test
test_cases = [
    ([], [], "empty"),
    (["Python", "Java"], ["Python", "Java"], "perfect"),
    (["Python", "Java", "SQL"], ["Python", "TensorFlow"], "partial"),
    (["Python"], [], "no prediction"),
    ([], ["Python"], "hallucination"),
]
for gold, pred, name in test_cases:
    m = compute_metrics(gold, pred)
    print(f"""{name:15s} P={m['precision']:.2f} R={m['recall']:.2f} F1={m['f1']:.2f} (tp={m['tp']} fp={m['fp']} fn={m['fn']})""")

## 3. Per-Category Breakdown

Aggregate F1 hides structure: a system can score perfectly on programming languages while missing every cloud skill, and the single headline number will not tell you. Breaking metrics down per category turns evaluation from a score into a diagnosis — it shows *where* the pipeline degrades, which is what you actually fix.

**What the code does:** each gold item is bucketed via `categories.get(item, "other")`, so `Docker` and `SQL` — absent from the category map — land in `other`. The gold loop counts tp/fn per bucket; the predicted loop counts fp. Running it reports: programming P=1.00 R=0.50 F1=0.67 (`Java` was missed), nlp and ml at 1.00, cloud at 0.00 (`AWS` missed), and `other` at 0.00 — both false positives (`Docker`, `SQL`) park in `other` because they were never part of the taxonomy.

**Try it:** treat the `other` bucket as a warning light — every fp parked there means the extractor is finding skills the evaluation schema does not even model.

In [ ]:
# Evaluate skill extraction per category
categories = {
    "Python": "programming",
    "Java": "programming",
    "NLP": "nlp",
    "TensorFlow": "ml",
    "AWS": "cloud",
}
gold = ["Python", "Java", "NLP", "TensorFlow", "AWS"]
predicted = ["Python", "NLP", "TensorFlow", "Docker", "SQL"]  # Docker and SQL are wrong

from collections import defaultdict
cat_metrics = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

for item in gold:
    cat = categories.get(item, "other")
    if item in predicted:
        cat_metrics[cat]["tp"] += 1
    else:
        cat_metrics[cat]["fn"] += 1

for item in predicted:
    cat = categories.get(item, "other")
    if item not in gold:
        cat_metrics[cat]["fp"] += 1

print("Per-category breakdown:")
for cat, m in sorted(cat_metrics.items()):
    p = m["tp"] / (m["tp"] + m["fp"]) if (m["tp"] + m["fp"]) > 0 else 0
    r = m["tp"] / (m["tp"] + m["fn"]) if (m["tp"] + m["fn"]) > 0 else 0
    f1 = 2*p*r/(p+r) if (p+r) > 0 else 0
    print(f"  {cat:12s} P={p:.2f} R={r:.2f} F1={f1:.2f}")

## Summary: Precision, recall, F1 are the standard NLP evaluation metrics. Track per-category for insights.

**Precision and recall are two views of the same error, and F1 forces you to balance them.**

A production resume parser needs both numbers because each hides what the other shows: high precision alone can be achieved by extracting nothing, high recall alone by extracting everything. The set-based `compute_metrics()` — tp/fp/fn from set operations — is the smallest implementation that gets the edge cases right, and the per-category breakdown is what turns a score into an action plan. With this machinery in hand, the next chapter generalizes from one-vs-rest skill matching to the full **confusion matrix** for multi-class section classification, where these same tp/fp/fn counts are laid out as a grid.